In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ============================================================
# 1. Load dataset
# ============================================================
FILE_NAME = "patient_list_with_ids"

# Read the excel file
df = pd.read_excel("../../"+FILE_NAME+".xlsx")

# Show the head
print(df.tail())

In [ ]:
# Extract side information from name column
def extract_side(name):
    name = str(name).upper().strip()

    if "SOL" in name:
        return "left"
    elif "SAĞ" in name or "SAG" in name:
        return "right"
    else:
        return "unknown"

In [ ]:
df["side"] = df["name"].apply(extract_side)

In [ ]:
# Convert Pederson to numeric
df["pederson"] = pd.to_numeric(df["pederson"], errors="coerce")

In [ ]:
# Keep only left and right records
df_lr = df[df["side"].isin(["left", "right"])].copy()
df_lr.tail()

In [ ]:
# Check how many records exist for each patient
patient_side_counts = df_lr.groupby("id")["side"].nunique()

# Keep only patients who have both left and right records
bilateral_patient_ids = patient_side_counts[patient_side_counts == 2].index

df_bilateral = df_lr[df_lr["id"].isin(bilateral_patient_ids)].copy()

print("Number of tooth-level records:", len(df))
print("Number of patients with both left and right records:", len(bilateral_patient_ids))

In [ ]:
# Create paired left-right table
paired = df_bilateral.pivot_table(
    index="id",
    columns="side",
    values=["pell_gregory", "pederson", "winter_angulation"],
    aggfunc="first"
)

# Flatten multi-level columns
paired.columns = [f"{variable}_{side}" for variable, side in paired.columns]

# Reset index
paired = paired.reset_index()

print(paired.head())

In [ ]:
# Save left and right records into separate Excel files
paired.to_excel("../../paired_list.xlsx", index=False)